In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/content/drive/MyDrive/DVA_capstone/master_dataset.csv')

In [4]:
df.shape

(2864948, 30)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2864948 entries, 0 to 2864947
Data columns (total 30 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   video_id                 object 
 1   title                    object 
 2   channel_id               object 
 3   channel_title            object 
 4   category_name            object 
 5   publish_time             object 
 6   publish_hour             int64  
 7   publish_day              object 
 8   trending_date            object 
 9   views                    int64  
 10  likes                    int64  
 11  comment_count            int64  
 12  comments_per_1000_views  float64
 13  dislikes                 int64  
 14  likes_ratio              float64
 15  engagement_rate          float64
 16  days_to_trend            int64  
 17  virality_score           float64
 18  tags                     object 
 19  tags_count               int64  
 20  has_description          bool   
 21  comments

## Category Dashboard

In [6]:
df['dislikes_ratio'] = df['dislikes'] / (df['likes'] + df['dislikes'] + 1)

In [7]:
category_df = (
    df.groupby(['country', 'category_name', 'trending_date', 'trend_speed'], as_index=False)
      .agg(
          total_views=('views', 'sum'),
          total_likes=('likes', 'sum'),
          total_comments=('comment_count', 'sum'),
          total_dislikes=('dislikes', 'sum'),
          video_count=('video_id', 'nunique'),
          avg_engagement_rate=('engagement_rate', 'mean'),
          avg_likes_per_video=('likes', 'mean'),
          avg_comments_per_video=('comment_count', 'mean'),
          avg_virality_score=('virality_score', 'mean'),
          avg_days_to_trend=('days_to_trend', 'mean'),
          avg_dislike_ratio=('dislikes_ratio', 'mean'),
          avg_views_log=('views_log', 'mean'),
          avg_virality_score_log=('virality_score_log', 'mean')
      )
      .sort_values(['country', 'category_name', 'trending_date'])
      .reset_index(drop=True)
)

In [8]:
category_df['views_growth_rate'] = (
    category_df.groupby(['country', 'category_name'])['total_views'].pct_change()
)

category_df['engagement_growth_rate'] = (
    category_df.groupby(['country', 'category_name'])['avg_engagement_rate'].pct_change()
)

In [9]:
category_kpis = (
    category_df.groupby(['country', 'category_name'], as_index=False)
      .agg(
          mean_views=('total_views', 'mean'),
          std_views=('total_views', 'std'),
          mean_engagement_rate=('avg_engagement_rate', 'mean'),
          std_engagement_rate=('avg_engagement_rate', 'std'),
          avg_views_growth=('views_growth_rate', 'mean'),
          median_views_growth=('views_growth_rate', 'median'),
          avg_engagement_growth=('engagement_growth_rate', 'mean')
      )
)

category_kpis['cv_views'] = category_kpis['std_views'] / category_kpis['mean_views']

category_kpis['cv_engagement_rate'] = (
    category_kpis['std_engagement_rate'] / category_kpis['mean_engagement_rate']
)

category_kpis['consistency_score'] = 1 / (1 + category_kpis['cv_engagement_rate'])

In [10]:
best_consistent = (
    category_kpis.sort_values(['country', 'cv_engagement_rate', 'category_name'])
                 .groupby('country', as_index=False)
                 .first()[['country', 'category_name', 'cv_engagement_rate', 'consistency_score']]
                 .rename(columns={'category_name': 'most_consistent_category'})
)

fastest_growing = (
    category_kpis.sort_values(['country', 'avg_views_growth', 'category_name'], ascending=[True, False, True])
                 .groupby('country', as_index=False)
                 .first()[['country', 'category_name', 'avg_views_growth', 'median_views_growth']]
                 .rename(columns={'category_name': 'fastest_growing_category'})
)

best_consistent['kpi_type'] = 'most_consistent_category'
fastest_growing['kpi_type'] = 'fastest_growing_category'

category_kpi_lookup = pd.merge(
    best_consistent,
    fastest_growing,
    on='country',
    how='outer'
)

category_df = category_df.merge(
    category_kpi_lookup,
    on='country',
    how='left'
)

category_df.drop(['kpi_type_x', 'kpi_type_y'], axis=1, inplace=True)

In [11]:
category_df.shape

(489558, 25)

In [12]:
category_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 489558 entries, 0 to 489557
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   country                   489558 non-null  object 
 1   category_name             489558 non-null  object 
 2   trending_date             489558 non-null  object 
 3   trend_speed               489558 non-null  object 
 4   total_views               489558 non-null  int64  
 5   total_likes               489558 non-null  int64  
 6   total_comments            489558 non-null  int64  
 7   total_dislikes            489558 non-null  int64  
 8   video_count               489558 non-null  int64  
 9   avg_engagement_rate       489558 non-null  float64
 10  avg_likes_per_video       489558 non-null  float64
 11  avg_comments_per_video    489558 non-null  float64
 12  avg_virality_score        489558 non-null  float64
 13  avg_days_to_trend         489558 non-null  f

In [13]:
category_df.to_csv('/content/drive/MyDrive/DVA_capstone/category_dataset.csv', index=False)

## Timing & Virality Dashboard


In [ ]:
timing_virality_df = (
    df.groupby(['country', 'publish_day', 'publish_hour'], as_index=False)
      .agg(
          total_views=('views', 'sum'),
          total_likes=('likes', 'sum'),
          total_comments=('comment_count', 'sum'),
          video_count=('video_id', 'nunique'),
          avg_engagement_rate=('engagement_rate', 'mean'),
          avg_days_to_trend=('days_to_trend', 'mean'),
          avg_virality_score=('virality_score', 'mean'),
          avg_views_per_video=('views', 'mean'),
          avg_views_log=('views_log', 'mean'),
          avg_virality_score_log=('virality_score_log', 'mean'),
          avg_trend_speed=('trend_speed', lambda x: x.astype(str).mode().iloc[0] if len(x.dropna()) else np.nan)
      )
)

In [ ]:
timing_virality_df.shape

(1848, 14)

In [ ]:
timing_virality_df

,country,publish_day,publish_hour,total_views,total_likes,total_comments,video_count,avg_engagement_rate,avg_days_to_trend,avg_virality_score,avg_views_per_video,avg_views_log,avg_virality_score_log,avg_trend_speed
0,Brazil,Friday,0,6565355002,307636357,17906066,509,0.071302,3.661192,5.008333e+05,1.996762e+06,13.095320,11.768358,Moderate
1,Brazil,Friday,1,1319785239,77016570,3497522,177,0.077704,3.188462,4.133544e+05,1.269024e+06,13.057974,11.850733,Moderate
2,Brazil,Friday,2,828891769,43638966,3058339,179,0.068655,2.987097,3.528614e+05,8.912815e+05,12.958951,11.791754,Moderate
3,Brazil,Friday,3,9569034675,726249751,166201039,222,0.079278,3.470926,1.969752e+06,6.869372e+06,13.466241,12.175326,Moderate
4,Brazil,Friday,4,29311642568,2041730751,269127357,420,0.084832,3.687053,2.757554e+06,1.048342e+07,14.883703,13.545208,Moderate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1843,US,Wednesday,19,4908877277,242451881,15637605,464,0.061457,3.458791,4.928509e+05,1.926561e+06,13.776831,12.406952,Moderate
1844,US,Wednesday,20,3363617591,160128475,9985344,451,0.058504,3.353336,3.572177e+05,1.411505e+06,13.610559,12.245396,Moderate
1845,US,Wednesday,21,5605227872,329557457,15999421,447,0.052928,3.571726,4.784819e+05,2.337459e+06,13.637843,12.241005,Moderate
1846,US,Wednesday,22,4002594209,223249424,15662544,435,0.056633,3.513871,4.129082e+05,1.708320e+06,13.590293,12.195793,Moderate


In [ ]:
timing_virality_df.to_csv('/content/drive/MyDrive/DVA_capstone/timing_virality_dataset.csv', index=False)